# Methodology decisions

This notebook is used to explain how the model developed over time and which methodology decisions were taken. Further discussion can be found in the full report. This is currently not in the order of the report and is merely workings out.

**Data sources used in this notebook:**
- `sudan_results.csv` - current set of all runs
- `ethiopia_results.csv` — Ethiopia/Tigray replication runs

In [1]:

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from transformers import AutoTokenizer

from processing import acled_events_processing as acled
from processing import food_prices_processing as food
from processing import rainfall_processing as rain
from processing.acled_text_processing import (
    check_max_tokens,
    remove_dates,
)
from utils.constants import ONSET_END_DATE, ONSET_START_DATE
from utils.data_prep import get_clean_combined_data

sudan = pd.read_csv("evaluation/sudan_results.csv").drop_duplicates(
    subset="run_id", keep="last"
)
ethiopia = pd.read_csv("evaluation/ethiopia_results.csv")


print(f"Sudan: {len(sudan)} runs")
print(f"Ethiopia: {len(ethiopia)} runs")


Sudan: 1120 runs
Ethiopia: 32 runs


# 1. Data
## 1.1 Abyei

Abyei is a disputed border region between Sudan and South Sudan, different organsiations have different collection methodologies in Abyei. In ACLED data, Abyei is included as part of Sudan at admin level 1, whereas in World Food Prices (WFP) it is included in the South Sudan dataset. The initial pipeline for WFP downloaded and combined both Sudan and South Sudan data to include Abyei. However, when it came to rainfall data, neither the Sudan or South Sudan data set contained information for Abyei. Meaning it would have to be calculated from the raw data instead of the rolling mean. The data would then be different for Abyei rainfall. 

Abyei accounts for a small share of overall conflict activity: 435 of 27,917 total events in the
raw ACLED pull (1.56%), spanning July 2017 to December 2025. Given this small share, the
cross-dataset inconsistency in which country's data source Abyei belongs to, and the missing
rainfall coverage that would have required a separate calculation method for this one region,
Abyei was excluded from the analysis for consistency across all data sources.

In [ ]:
raw_acled = pd.read_csv("data/acled/acled_sudan.csv")
abyei_events = raw_acled[
    raw_acled["admin1"].str.lower().str.contains("abyei", na=False)
]
print(
    f"Abyei events in raw data: {len(abyei_events)} of {len(raw_acled)} total ({len(abyei_events) / len(raw_acled) * 100:.2f}%)"
)
print(
    f"Date range: {abyei_events['event_date'].min()} to {abyei_events['event_date'].max()}"
)

## 1.2 Text corpus
The ACLED notes columns contains further details about the event reported. These notes vary in length and details. 
ConfliBERT only takes 512 tokens. The original idea was to concatenate each region-month set of notes columns into one long narrative but that would have gone over the amount of tokens meaning that important information might get missed. Instead each event was tokenised and an average for each month was created. Then I looked at whether it was just the conflict text that was contributing to the prediction by adding a conflict vs non-conflict flag

Unlike the structural features, text embeddings were lagged by a single month rather than averaged over a rolling window. This was a deliberate choice: a rolling average would dilute
recent narrative shifts with older, less relevant text, and would additionally be distorted by the zero-vector placeholder used for months with no ACLED events, since an embedding of zero does not carry the same meaningful interpretation as a genuine zero conflict-event count.


## 1.3 Missing values
The data set was expanded to have a full region-month index, this resulted in some missing values, for each of the data sets this was treated differently.

**ACLED events**

Missing region-months were filled with 0. Here, a missing value means no conflict
events were recorded so this is a genuine signal. 

**Food prices**

Missing region-months forward-filled (last known price carried forward until a new reading).

Risk: stale signal during rapid change. Confirmed this actually happens: Khartoum has no price
data for any commodity in April 2023, the month fighting broke out there. Forward-fill would carry
its last pre-war price forward instead of reflecting the shock.

Leading NaNs (no price ever recorded) left as NaN for XGBoost to handle natively.


**Rainfall**

Unlike food prices, rainfall required no missing-value filling in the final feature set. The raw HDX rainfall data has complete coverage across all 18 regions from the 1980s onward, with the only gap in the source (144 rows, evenly split across regions) occurring in January 1981, decades before the study period. The one-month warm-up buffer built into the padding logic absorbs the NaN introduced by the 1-month lag applied to every region's series, so the final rainfall_3m_anomaly feature contains zero missing values across all 1,728 rows.

In [ ]:
# ACLED
# Note: this data includes the 6-month warm up period
acled_df, acled_predictor_cols, _ = acled.get_clean_data(
    k=1.75, event_col="sub_event_type"
)
n_total_acled = len(acled_df)
pct_zero = (acled_df[acled_predictor_cols] == 0).mean().mean() * 100
print(
    f"\nACLED: {n_total_acled} rows, {pct_zero:.1f}% of predictor values are 0 (genuine no-event signal)"
)
print()

# Food prices
all_regions = acled_df["region"].unique()
all_months = pd.period_range(
    acled_df["year_month"].min(), acled_df["year_month"].max(), freq="M"
)

food_df, food_predictor_cols = food.get_clean_data(
    all_regions=all_regions, all_months=all_months
)
n_total_food = len(food_df)
print(f"Food prices: {n_total_food} rows")
for col in food_predictor_cols:
    n_missing = food_df[col].isna().sum()
    print(
        f"  {col}: {n_missing} missing ({n_missing / n_total_food * 100:.1f}%) - leading gaps before first recorded price"
    )
print()

# Rainfall
rain_df, rain_predictor_cols = rain.get_clean_data(
    all_regions=all_regions, all_months=all_months
)
n_total_rain = len(rain_df)
for col in rain_predictor_cols:
    n_missing = rain_df[col].isna().sum()
    print(
        f"Rainfall: {col}: {n_missing} missing of {n_total_rain} rows ({n_missing / n_total_rain * 100:.1f}%)"
    )

# 2. Setting the target - choosing the k escalation threshold
## 2.1 Rejecting k=0.25

`k` controls the escalation threshold — a lower k means a looser threshold (escalation is easier to trigger).

**Issue**

Raw AUPR favours the lowest `k`, but that is misleading for conflict prediction. The model was catching 100% of true positives by default rather than through genuine skill. Domain knowledge also matters here - in conflict forecasting, you want a model that's sensitive to real escalations without flagging every small, ordinary shift in conflict as significant.

Three values of `k` (0.25, 0.5, 1.0) were tested in the initial sweep, measuring both the true onset-window prevalence of escalation and the rate at which the F1-optimal threshold collapsed to predicting positive for nearly every region-month.

`k`=0.25 had the highest prevalence (42.1%) and the highest collapse rate (75.0%), making it a poor definition of true escalation. `k`=0.5 was somewhat better (37.0% prevalence, 57.2% collapse) but still substantially collapsed. This motivated dropping `k`=0.25 from further consideration, and investigating the collapse behaviour more closely - covered in Section 1.2.

In [ ]:
prevalence_records = []
for k_test in [0.25, 0.5, 1]:
    model_data, predictor_cols = get_clean_combined_data(
        data_sources=[],
        k=k_test,
        event_col="sub_event_type",
        conflict_only_embeddings=True,
    )
    onset_slice = model_data[
        (model_data["year_month"] >= pd.Period(ONSET_START_DATE, freq="M"))
        & (model_data["year_month"] <= pd.Period(ONSET_END_DATE, freq="M"))
    ]
    n_total = len(onset_slice)
    n_escalations = int(onset_slice["target_escalation"].sum())
    prevalence_records.append(
        {
            "k": k_test,
            "onset_prevalence_pct": round(n_escalations / n_total * 100, 1),
            "n_escalations": n_escalations,
            "n_onset_rows": n_total,
        }
    )
prevalence_df_early = pd.DataFrame(prevalence_records).set_index("k")

sudan_pre_fix = sudan[sudan["threshold_fix_applied"] == False]

collapse_rate_pre_fix = sudan_pre_fix.groupby("k")["onset_recall_class1"].apply(
    lambda x: (x == 1.0).mean()
)

k_summary_1_1 = pd.DataFrame(
    {
        "mean_onset_aupr": sudan_pre_fix.groupby("k")["onset_aupr"].mean(),
        "max_onset_aupr": sudan_pre_fix.groupby("k")["onset_aupr"].max(),
        "collapse_rate": collapse_rate_pre_fix,
    }
)
k_summary_1_1 = k_summary_1_1.join(prevalence_df_early[["onset_prevalence_pct"]])
k_summary_1_1.round(3)

In [ ]:
ks = k_summary_1_1.index.astype(str)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Positive-class prevalence by k", "Threshold-collapse rate (%)"),
)

fig.add_trace(
    go.Bar(
        x=ks,
        y=k_summary_1_1["onset_prevalence_pct"],
        marker_color="#898781",
        name="Prevalence",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=ks,
        y=k_summary_1_1["collapse_rate"] * 100,
        marker_color="#c0392b",
        name="Collapse rate",
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="k", row=1, col=1)
fig.update_xaxes(title_text="k", row=1, col=2)
fig.update_yaxes(title_text="% of onset rows that were escalations", row=1, col=1)
fig.update_yaxes(title_text="% of runs with recall=1.0", row=1, col=2)

fig.update_layout(
    showlegend=False,
    width=900,
    height=450,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(t=120),
    title={
        "text": "k=0.25 had the highest prevalence and the highest collapse rate,<br>before the threshold-collapse fix was applied",
        "y": 0.9,
        "yanchor": "top",
    },
)
fig.show()

## 2.2 The threshold-collapse bug and fix

**Bug**

In the initial set of runs, `optimal_threshold = thresholds[np.argmax(f1_scores)]` picked the lowest tied threshold whenever F1 plateaued, producing "predict everything positive" models. This was discovered because many runs had recall=1 and precision equal to the actual proportion of escalations in the data.

**Fix**

Instead, the model now picks the highest threshold among those tied for the best F1 score, making it more conservative and less likely to default to predicting everything positive.

```python
max_f1 = f1_scores.max()
tied_indices = np.flatnonzero(f1_scores == max_f1)
optimal_threshold = thresholds[tied_indices[-1]]
```

**Effect of the fix**

Holding `k` constant (0.5 and 1.0 - as 0.25 was dropped), the collapse rate fell from 50.0% to 40.6%. This is an improvement, but it did not eliminate the issue at the looser threshold. Therefore further work was required to investigate raising `k`, covered in Section 1.3.

In [ ]:
pre_threshold_fix = sudan[sudan["threshold_fix_applied"] == False]
threshold_fix = sudan[sudan["threshold_fix_applied"] == True]

pre_fix_shared = pre_threshold_fix[pre_threshold_fix["k"].isin([0.5, 1.0])]
post_fix_shared = threshold_fix[threshold_fix["k"].isin([0.5, 1.0])]

pre_fix_collapse_rate = (pre_fix_shared["onset_recall_class1"] == 1.0).mean()
post_fix_collapse_rate = (post_fix_shared["onset_recall_class1"] == 1.0).mean()

print(f"Pre-fix collapse rate (k=0.5/1.0 only): {pre_fix_collapse_rate:.1%}")
print(f"Post-fix collapse rate (k=0.5/1.0 only): {post_fix_collapse_rate:.1%}")

## 2.3 Selecting k among post-fix thresholds

With the corrected threshold logic in place, `k` was tested more broadly (0.5 to 2.5) to find the strictest threshold at which collapse is effectively resolved without giving up genuine model performance.

`k` was chosen once across the whole set of model combinations rather than separately for each one, since testing it against every food, rain, text and event type configuration gives a far larger and more reliable picture than tuning it against a single model would.

`k`=1.75 was chosen mainly because the collapse rate drops to 2.5% at this point, and because a higher `k` pushes the target closer to what escalation actually looks like in practice. Civil conflict onset in the literature is estimated at somewhere between 1.3 and 2.1 events per 100 country-years depending on region (Fearon and Laitin, 2003), so a target that flags conflict as often as 40-70% of the time, which is what we see at the lower `k` values, does not reflect how rare real escalation is. k=1.75 keeps prevalence low without losing model performance. However Fearon and Laitin's figure measures a discrete, national, once-in-a-conflict event, while this model measures a monthly regional deviation from a rolling baseline, which is a naturally more frequent kind of event.

It is worth noting that `k`=1.65 actually produced a slightly higher onset AUPR (0.4128 versus 0.4043), but it also came with double the collapse rate (5.0% versus 2.5%). Given the small-ish gain and real life esclations rates, 1.75 was chosen. 

In [ ]:
prevalence_records = []
for k_test in [0.5, 1.0, 1.25, 1.5, 1.6, 1.65, 1.75, 2.0, 2.5]:
    model_data, predictor_cols = get_clean_combined_data(
        data_sources=[],
        k=k_test,
        event_col="sub_event_type",
        conflict_only_embeddings=True,
    )
    onset_slice = model_data[
        (model_data["year_month"] >= pd.Period(ONSET_START_DATE, freq="M"))
        & (model_data["year_month"] <= pd.Period(ONSET_END_DATE, freq="M"))
    ]
    n_total = len(onset_slice)
    n_escalations = int(onset_slice["target_escalation"].sum())
    prevalence_records.append(
        {
            "k": k_test,
            "onset_prevalence_pct": round(n_escalations / n_total * 100, 1),
            "n_escalations": n_escalations,
            "n_onset_rows": n_total,
        }
    )
prevalence_df_full = pd.DataFrame(prevalence_records).set_index("k")

In [ ]:
threshold_fix = sudan[sudan["threshold_fix_applied"] == True]
genuine = threshold_fix[
    threshold_fix["onset_recall_class1"] <= 0.9
]  # Where the model is being at least a bit selective

collapse_rate = threshold_fix.groupby("k")["onset_recall_class1"].apply(
    lambda x: (x > 0.9).mean()
)

k_summary_1_3 = pd.DataFrame(
    {
        "mean_onset_aupr": threshold_fix.groupby("k")["onset_aupr"].mean(),
        "best_genuine_onset_aupr": genuine.groupby("k")["onset_aupr"].max(),
        "collapse_rate": collapse_rate,
    }
)
k_summary_1_3 = k_summary_1_3.join(prevalence_df_full[["onset_prevalence_pct"]])
k_summary_1_3.round(3)

In [ ]:
ks = k_summary_1_3.index.astype(str)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Threshold-collapse rate (%)", "Best genuine onset AUPR"),
)

fig.add_trace(
    go.Bar(
        x=ks,
        y=k_summary_1_3["collapse_rate"] * 100,
        marker_color="#c0392b",
        name="Collapse rate",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=ks,
        y=k_summary_1_3["best_genuine_onset_aupr"],
        marker_color="#2a78d6",
        name="Best genuine AUPR",
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="k", row=1, col=1)
fig.update_xaxes(title_text="k", row=1, col=2)
fig.update_yaxes(title_text="% of runs with recall>0.9", row=1, col=1)
fig.update_yaxes(title_text="onset AUPR", row=1, col=2)

fig.update_layout(
    showlegend=False,
    width=900,
    height=450,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(t=120),
    title={
        "text": "Post-fix: collapse falls as k increases, while higher AUPR performance is<br>retained at k=1.75",
        "y": 0.9,
        "yanchor": "top",
    },
)
fig.show()

# 3. N splits

`n_splits` controls the number of expanding-window cross-validation folds used during
hyperparameter search (4 or 5, tested throughout the project).

Comparison at `k`=1.75 shows `n_splits`=5 with a marginally higher mean and
maximum onset AUPR than `n_splits`=4 (mean 0.3348 vs 0.3204, max 0.4043 vs 0.3977), though the
difference is small.

`n_splits`=5 was adopted for Model A and Model B, giving
identical cross-validation folds across the comparison, with confirmation that
this choice does not materially affect the reported results.

In [ ]:
k175 = sudan[(sudan["k"] == 1.75) & (sudan["threshold_fix_applied"] == True)]

k175.groupby("n_splits")["onset_aupr"].agg(["mean", "max", "count"]).round(4)

# 4. Averaging monthly embeddings
At the proposal stage, it was considered to concatenate all notes within a region-month into a single ordered text block. However, testing this approach showed it increased the average length to 768 tokens, well above the model's limit. In months with higher event volume, this combined note would exceed the token limit further still, risking the loss of important information during the embedding phase. Instead, embeddings were generated for each individual event and then averaged at the region-month level. However, this averaging approach does discard the temporal ordering of events within the month and can dilute the signal of a severe incident when averaged.

In [ ]:
model_name = "eventdata-utd/ConfliBERT-scr-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


df_regex = remove_dates(raw_acled)
check_max_tokens(tokenizer, df_regex)

df_regex_sorted = df_regex.sort_values(["admin1", "event_date"])


df_region_month_notes = (
    df_regex_sorted.groupby(["admin1", "year_month"])["notes_cleaned"]
    .apply(lambda notes: " ".join(notes.dropna()))
    .reset_index()
    .rename(columns={"admin1": "region"})
)
check_max_tokens(tokenizer, df_region_month_notes)

